In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
import plotly.graph_objects as go
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["font.size"] = 11

In [ ]:
df = pd.read_csv("DailyDelhiClimate.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df.info()
df.head()

In [ ]:
df.describe().T

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(df["date"], df["meantemp"], linewidth=1.5)
axes[0, 0].set_title("Mean Temperature")
axes[0, 0].set_xlabel("Date")
axes[0, 0].set_ylabel("Temperature")

axes[0, 1].plot(df["date"], df["humidity"], linewidth=1.5)
axes[0, 1].set_title("Humidity")
axes[0, 1].set_xlabel("Date")
axes[0, 1].set_ylabel("Humidity")

axes[1, 0].plot(df["date"], df["wind_speed"], linewidth=1.5)
axes[1, 0].set_title("Wind Speed")
axes[1, 0].set_xlabel("Date")
axes[1, 0].set_ylabel("Wind Speed")

axes[1, 1].plot(df["date"], df["meanpressure"], linewidth=1.5)
axes[1, 1].set_title("Mean Pressure")
axes[1, 1].set_xlabel("Date")
axes[1, 1].set_ylabel("Pressure")

plt.tight_layout()
plt.show()

In [ ]:
def loss_function_xy(x, y):
    return (
        0.18 * (x**2 + y**2)
        + 2.5 * tf.sin(1.15 * x) * tf.sin(0.9 * y)
        + 0.035 * (x**2 - y**2)**2
    )

def gradient_at(point):
    p = tf.Variable(point, dtype=tf.float32)
    with tf.GradientTape() as tape:
        loss = loss_function_xy(p[0], p[1])
    grad = tape.gradient(loss, p)
    return float(loss.numpy()), grad.numpy().astype(float)

xv = np.linspace(-6, 6, 260)
yv = np.linspace(-6, 6, 260)
X, Y = np.meshgrid(xv, yv)
Z = np.zeros_like(X)

for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i, j] = float(loss_function_xy(tf.constant(X[i, j]), tf.constant(Y[i, j])).numpy())

np.nanmin(Z), np.nanmax(Z)

In [ ]:
fig = go.Figure(
    data=[
        go.Surface(
            x=X,
            y=Y,
            z=Z,
            colorscale="Turbo",
            colorbar=dict(title="Loss"),
            contours={
                "z": {
                    "show": True,
                    "usecolormap": True,
                    "highlightcolor": "white",
                    "project_z": True
                }
            }
        )
    ]
)

fig.update_layout(
    title="3D Loss Surface",
    scene=dict(
        xaxis_title="w1",
        yaxis_title="w2",
        zaxis_title="Loss",
        camera=dict(eye=dict(x=1.55, y=1.55, z=1.15))
    ),
    width=1000,
    height=750
)
fig.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
levels = np.linspace(np.percentile(Z, 3), np.percentile(Z, 92), 35)
contour = ax.contourf(X, Y, Z, levels=levels, cmap="turbo")
lines = ax.contour(X, Y, Z, levels=levels[::3], colors="white", alpha=0.28, linewidths=0.7)
ax.clabel(lines, inline=True, fontsize=8, fmt="%.1f")
plt.colorbar(contour, ax=ax, label="Loss")
ax.set_title("Contour View of the Loss Surface")
ax.set_xlabel("w1")
ax.set_ylabel("w2")
ax.set_aspect("equal")
plt.show()

In [ ]:
def run_sgd(start, lr=0.035, steps=90):
    point = np.array(start, dtype=float)
    path = [point.copy()]
    losses = []
    gradients = []

    for _ in range(steps):
        loss, grad = gradient_at(point)
        losses.append(loss)
        gradients.append(grad.copy())
        point = point - lr * grad
        path.append(point.copy())

    return np.array(path), np.array(losses), np.array(gradients)

def run_momentum(start, lr=0.035, beta=0.90, steps=90):
    point = np.array(start, dtype=float)
    velocity = np.zeros(2, dtype=float)
    path = [point.copy()]
    losses = []
    gradients = []
    velocities = []

    for _ in range(steps):
        loss, grad = gradient_at(point)
        velocity = beta * velocity + grad
        losses.append(loss)
        gradients.append(grad.copy())
        velocities.append(velocity.copy())
        point = point - lr * velocity
        path.append(point.copy())

    return np.array(path), np.array(losses), np.array(gradients), np.array(velocities)

start = np.array([5.2, 4.4])

sgd_path, sgd_losses, sgd_grads = run_sgd(start)
mom_path, mom_losses, mom_grads, mom_vel = run_momentum(start)

sgd_path[-1], mom_path[-1]

In [ ]:
fig, ax = plt.subplots(figsize=(13, 10))
levels = np.linspace(np.percentile(Z, 3), np.percentile(Z, 92), 40)
cf = ax.contourf(X, Y, Z, levels=levels, cmap="turbo")
ax.contour(X, Y, Z, levels=levels[::3], colors="white", alpha=0.24, linewidths=0.7)
ax.plot(sgd_path[:, 0], sgd_path[:, 1], linewidth=2.5, label="SGD")
ax.plot(mom_path[:, 0], mom_path[:, 1], linewidth=2.5, label="Momentum")
ax.scatter(*start, s=100, marker="*", label="Start")
ax.scatter(*sgd_path[-1], s=100, marker="X", label="SGD End")
ax.scatter(*mom_path[-1], s=100, marker="X", label="Momentum End")
ax.set_title("SGD vs Momentum: Complete Optimization Paths")
ax.set_xlabel("w1")
ax.set_ylabel("w2")
ax.legend()
ax.set_aspect("equal")
plt.colorbar(cf, ax=ax, label="Loss")
plt.show()

In [ ]:
frames = max(len(sgd_path), len(mom_path))

fig, axes = plt.subplots(1, 2, figsize=(17, 8))
levels = np.linspace(np.percentile(Z, 3), np.percentile(Z, 92), 40)

for ax in axes:
    ax.contourf(X, Y, Z, levels=levels, cmap="turbo")
    ax.contour(X, Y, Z, levels=levels[::3], colors="white", alpha=0.24, linewidths=0.7)
    ax.scatter(*start, s=100, marker="*", zorder=5)
    ax.set_xlim(xv.min(), xv.max())
    ax.set_ylim(yv.min(), yv.max())
    ax.set_aspect("equal")

sgd_line, = axes[0].plot([], [], linewidth=3)
sgd_point, = axes[0].plot([], [], marker="o", markersize=9)
sgd_grad, = axes[0].plot([], [], linewidth=2)

mom_line, = axes[1].plot([], [], linewidth=3)
mom_point, = axes[1].plot([], [], marker="o", markersize=9)
mom_grad, = axes[1].plot([], [], linewidth=2)

axes[0].set_title("SGD")
axes[1].set_title("Momentum")

for ax in axes:
    ax.set_xlabel("w1")
    ax.set_ylabel("w2")

def init():
    sgd_line.set_data([], [])
    sgd_point.set_data([], [])
    sgd_grad.set_data([], [])
    mom_line.set_data([], [])
    mom_point.set_data([], [])
    mom_grad.set_data([], [])
    fig.suptitle("Optimization Step 0", fontsize=18)
    return sgd_line, sgd_point, sgd_grad, mom_line, mom_point, mom_grad

def update(frame):
    s = min(frame, len(sgd_path) - 1)
    m = min(frame, len(mom_path) - 1)

    sgd_line.set_data(sgd_path[:s+1, 0], sgd_path[:s+1, 1])
    sgd_point.set_data([sgd_path[s, 0]], [sgd_path[s, 1]])

    sg_loss, sg = gradient_at(sgd_path[s])
    sg_end = sgd_path[s] - 0.12 * sg
    sgd_grad.set_data([sgd_path[s, 0], sg_end[0]], [sgd_path[s, 1], sg_end[1]])

    mom_line.set_data(mom_path[:m+1, 0], mom_path[:m+1, 1])
    mom_point.set_data([mom_path[m, 0]], [mom_path[m, 1]])

    mm_loss, mg = gradient_at(mom_path[m])
    mg_end = mom_path[m] - 0.12 * mg
    mom_grad.set_data([mom_path[m, 0], mg_end[0]], [mom_path[m, 1], mg_end[1]])

    fig.suptitle(
        f"Optimization Step {frame}   |   SGD Loss: {sg_loss:.4f}   |   Momentum Loss: {mm_loss:.4f}",
        fontsize=17
    )
    return sgd_line, sgd_point, sgd_grad, mom_line, mom_point, mom_grad

anim = FuncAnimation(fig, update, frames=frames, init_func=init, interval=90, blit=False, repeat=True)
plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(16, 8))
ax1 = fig.add_subplot(121, projection="3d")
ax2 = fig.add_subplot(122, projection="3d")

stride = 5
Xs = X[::stride, ::stride]
Ys = Y[::stride, ::stride]
Zs = Z[::stride, ::stride]

for ax, title in [(ax1, "SGD"), (ax2, "Momentum")]:
    ax.plot_surface(Xs, Ys, Zs, cmap="turbo", alpha=0.62, linewidth=0, antialiased=True)
    ax.set_xlabel("w1")
    ax.set_ylabel("w2")
    ax.set_zlabel("Loss")
    ax.set_title(title)
    ax.view_init(elev=36, azim=-55)

sgd_3d, = ax1.plot([], [], [], linewidth=4)
sgd_3d_point, = ax1.plot([], [], [], marker="o", markersize=8)

mom_3d, = ax2.plot([], [], [], linewidth=4)
mom_3d_point, = ax2.plot([], [], [], marker="o", markersize=8)

def loss_np(point):
    return float(loss_function_xy(tf.constant(point[0]), tf.constant(point[1])).numpy())

def init3d():
    for artist in [sgd_3d, sgd_3d_point, mom_3d, mom_3d_point]:
        artist.set_data_3d([], [], [])
    fig.suptitle("3D Optimization Animation — Step 0", fontsize=17)
    return sgd_3d, sgd_3d_point, mom_3d, mom_3d_point

def update3d(frame):
    s = min(frame, len(sgd_path) - 1)
    m = min(frame, len(mom_path) - 1)

    sxyz = np.array([loss_np(p) for p in sgd_path[:s+1]])
    mxyz = np.array([loss_np(p) for p in mom_path[:m+1]])

    sgd_3d.set_data(sgd_path[:s+1, 0], sgd_path[:s+1, 1])
    sgd_3d.set_3d_properties(sxyz)

    sgd_3d_point.set_data([sgd_path[s, 0]], [sgd_path[s, 1]])
    sgd_3d_point.set_3d_properties([sxyz[-1]])

    mom_3d.set_data(mom_path[:m+1, 0], mom_path[:m+1, 1])
    mom_3d.set_3d_properties(mxyz)

    mom_3d_point.set_data([mom_path[m, 0]], [mom_path[m, 1]])
    mom_3d_point.set_3d_properties([mxyz[-1]])

    fig.suptitle(f"3D Optimization Animation — Step {frame}", fontsize=17)
    return sgd_3d, sgd_3d_point, mom_3d, mom_3d_point

anim3d = FuncAnimation(fig, update3d, frames=frames, init_func=init3d, interval=100, blit=False, repeat=True)
plt.close(fig)
HTML(anim3d.to_jshtml())

In [ ]:
comparison = pd.DataFrame({
    "step": np.arange(1, min(len(sgd_losses), len(mom_losses)) + 1),
    "SGD Loss": sgd_losses,
    "Momentum Loss": mom_losses
})

fig, ax = plt.subplots(figsize=(13, 7))
ax.plot(comparison["step"], comparison["SGD Loss"], linewidth=2.5, label="SGD")
ax.plot(comparison["step"], comparison["Momentum Loss"], linewidth=2.5, label="Momentum")
ax.set_yscale("symlog", linthresh=1e-3)
ax.set_xlabel("Optimization Step")
ax.set_ylabel("Loss")
ax.set_title("Loss vs Optimization Step")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

summary = pd.DataFrame({
    "Optimizer": ["SGD", "Momentum"],
    "Initial Loss": [sgd_losses[0], mom_losses[0]],
    "Final Loss": [sgd_losses[-1], mom_losses[-1]],
    "Best Loss": [np.min(sgd_losses), np.min(mom_losses)],
    "Final w1": [sgd_path[-1, 0], mom_path[-1, 0]],
    "Final w2": [sgd_path[-1, 1], mom_path[-1, 1]]
})
summary

In [ ]:
velocity_norm = np.linalg.norm(mom_vel, axis=1)
gradient_norm = np.linalg.norm(mom_grads, axis=1)

fig, ax = plt.subplots(figsize=(13, 7))
ax.plot(gradient_norm, linewidth=2.3, label="Gradient norm")
ax.plot(velocity_norm, linewidth=2.3, label="Momentum velocity norm")
ax.set_xlabel("Optimization Step")
ax.set_ylabel("Magnitude")
ax.set_title("Gradient vs Accumulated Momentum Velocity")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

In [ ]:
work = df.copy()
work["year"] = work["date"].dt.year
work["month"] = work["date"].dt.month
work["dayofyear"] = work["date"].dt.dayofyear
work["sin_day"] = np.sin(2 * np.pi * work["dayofyear"] / 365.25)
work["cos_day"] = np.cos(2 * np.pi * work["dayofyear"] / 365.25)

features = ["humidity", "wind_speed", "meanpressure", "sin_day", "cos_day"]
target = "meantemp"

X_data = work[features].values
y_data = work[target].values

split_index = int(len(work) * 0.8)

X_train = X_data[:split_index]
X_test = X_data[split_index:]
y_train = y_data[:split_index]
y_test = y_data[split_index:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape

In [ ]:
def build_ann():
    model = keras.Sequential([
        keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1)
    ])
    return model

sgd_model = build_ann()
momentum_model = build_ann()

sgd_model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.01, momentum=0.0, clipnorm=1.0),
    loss="mse",
    metrics=[keras.metrics.MeanAbsoluteError()]
)

momentum_model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.001, momentum=0.9, clipnorm=1.0),
    loss="mse",
    metrics=[keras.metrics.MeanAbsoluteError()]
)

sgd_history = sgd_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.15,
    epochs=120,
    batch_size=32,
    verbose=0
)

momentum_history = momentum_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.15,
    epochs=120,
    batch_size=32,
    verbose=0
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

axes[0].plot(sgd_history.history["loss"], linewidth=2.3, label="SGD")
axes[0].plot(momentum_history.history["loss"], linewidth=2.3, label="Momentum")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(sgd_history.history["val_loss"], linewidth=2.3, label="SGD")
axes[1].plot(momentum_history.history["val_loss"], linewidth=2.3, label="Momentum")
axes[1].set_title("Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MSE")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
sgd_pred = sgd_model.predict(X_test_scaled).ravel()
momentum_pred = momentum_model.predict(X_test_scaled).ravel()

metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R²"],
    "SGD": [
        mean_absolute_error(y_test, sgd_pred),
        np.sqrt(mean_squared_error(y_test, sgd_pred)),
        r2_score(y_test, sgd_pred)
    ],
    "Momentum": [
        mean_absolute_error(y_test, momentum_pred),
        np.sqrt(mean_squared_error(y_test, momentum_pred)),
        r2_score(y_test, momentum_pred)
    ]
})

metrics

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
test_dates = work["date"].iloc[split_index:]
ax.plot(test_dates, y_test, linewidth=2.2, label="Actual")
ax.plot(test_dates, sgd_pred, linewidth=1.8, label="SGD Prediction")
ax.plot(test_dates, momentum_pred, linewidth=1.8, label="Momentum Prediction")
ax.set_title("Delhi Mean Temperature — Actual vs ANN Predictions")
ax.set_xlabel("Date")
ax.set_ylabel("Mean Temperature")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

mn = min(y_test.min(), sgd_pred.min(), momentum_pred.min())
mx = max(y_test.max(), sgd_pred.max(), momentum_pred.max())

axes[0].scatter(y_test, sgd_pred, alpha=0.55, s=35)
axes[0].plot([mn, mx], [mn, mx], linewidth=2)
axes[0].set_title("SGD: Actual vs Predicted")
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")

axes[1].scatter(y_test, momentum_pred, alpha=0.55, s=35)
axes[1].plot([mn, mx], [mn, mx], linewidth=2)
axes[1].set_title("Momentum: Actual vs Predicted")
axes[1].set_xlabel("Actual")
axes[1].set_ylabel("Predicted")

plt.tight_layout()
plt.show()

In [ ]:
sgd_train_loss = np.array(sgd_history.history["loss"])
mom_train_loss = np.array(momentum_history.history["loss"])

fig, ax = plt.subplots(figsize=(13, 7))
ax.set_xlim(1, len(sgd_train_loss))
ax.set_ylim(
    min(sgd_train_loss.min(), mom_train_loss.min()) * 0.9,
    max(sgd_train_loss.max(), mom_train_loss.max()) * 1.05
)
ax.set_xlabel("Epoch")
ax.set_ylabel("Training MSE")
ax.set_title("ANN Training: SGD vs Momentum")

line1, = ax.plot([], [], linewidth=3, label="SGD")
line2, = ax.plot([], [], linewidth=3, label="Momentum")
point1, = ax.plot([], [], marker="o", markersize=7)
point2, = ax.plot([], [], marker="o", markersize=7)
ax.legend()
ax.grid(alpha=0.25)

def init_ann():
    line1.set_data([], [])
    line2.set_data([], [])
    point1.set_data([], [])
    point2.set_data([], [])
    return line1, line2, point1, point2

def update_ann(frame):
    n = frame + 1
    xs = np.arange(1, n + 1)
    line1.set_data(xs, sgd_train_loss[:n])
    line2.set_data(xs, mom_train_loss[:n])
    point1.set_data([n], [sgd_train_loss[n - 1]])
    point2.set_data([n], [mom_train_loss[n - 1]])
    ax.set_title(
        f"ANN Training — Epoch {n}   |   SGD: {sgd_train_loss[n-1]:.4f}   |   Momentum: {mom_train_loss[n-1]:.4f}"
    )
    return line1, line2, point1, point2

ann_anim = FuncAnimation(fig, update_ann, frames=len(sgd_train_loss), init_func=init_ann, interval=75, blit=False, repeat=True)
plt.close(fig)
HTML(ann_anim.to_jshtml())